# G11 — Gradient Attribution: Skip Path vs Prototype Path

**Goal:** Directly measure how much the decoder's output is driven by
encoder/skip-connection features vs prototype heatmap modulation.

**bypass_ratio** = `||∂logits/∂feat|| / (||∂logits/∂feat|| + ||∂logits/∂heatmap||)`

- bypass_ratio → 1: decoder relies on encoder/skip features (prototype bypassed)
- bypass_ratio → 0: decoder relies on prototype heatmap
- no-skip (9a, 9b): bypass_ratio = 0 by construction

**Core comparison:** same level configuration, skip vs no-skip:
- L4 only: Stage 8A (skip) vs 9a (no-skip, structural 0)
- L3+L4:   Stage 29 (skip) vs 9b (no-skip, structural 0)

**Additional:** Stage 34b (ALC) to check if regularisation reduces bypass.

**Output:** `results/v11/gradient_attribution_*.csv`

## 0. Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.models.proto_seg_net import ProtoSegNet
from src.data.mmwhs_dataset import MMWHSSliceDataset
from torch.utils.data import DataLoader

# ── Config ────────────────────────────────────────────────────────────────────
MODELS = {
    "stage8a":  "../checkpoints/proto_seg_ct_abl_a.pth",           # skip, L4 only
    "stage29":  "../checkpoints/proto_seg_ct_l3l4_warmstart.pth",  # skip, L3+L4
    "stage34b": "../checkpoints/proto_seg_ct_l3l4_alc_l3only.pth", # skip, L3+L4, ALC
}
DATA_DIR   = "../data/pack/processed_data"
MODALITY   = "ct"
N_SLICES   = 50  # same as Faithfulness eval
FG_CLASSES = list(range(1, 8))  # skip background (class 0)
CLASS_NAMES = ["BG", "LV", "RV", "LA", "RA", "Myo", "Aorta", "PA"]

OUT_DIR = Path("../results/v11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = (
    torch.device("mps")  if torch.backends.mps.is_available()  else
    torch.device("cuda") if torch.cuda.is_available()           else
    torch.device("cpu")
)
print(f"Device: {DEVICE}")
print(f"Output: {OUT_DIR.resolve()}")

## 1. Load Data

In [ ]:
test_ds = MMWHSSliceDataset(DATA_DIR, modality=MODALITY, split="test", preload=True)
indices = list(range(min(N_SLICES, len(test_ds))))
subset  = torch.utils.data.Subset(test_ds, indices)
loader  = DataLoader(subset, batch_size=1, shuffle=False, num_workers=0)
print(f"Evaluating on {len(subset)} slices")

## 2. Attribution Function

Temporarily replaces `model.forward` to intercept `feat[l]` and `agg_heatmap[l]`
as detached leaf tensors with `requires_grad=True`, then backtracks to measure
how much gradient flows through each path.

In [ ]:
def compute_attribution_for_slice(model, x, target_class):
    """
    Returns {level: {grad_feat, grad_heatmap, bypass_ratio}}.
    bypass_ratio = ||grad_feat|| / (||grad_feat|| + ||grad_heatmap||)
    """
    model.eval()
    x = x.to(DEVICE)

    feat_tensors    = {}
    heatmap_tensors = {}
    original_forward = model.forward

    def instrumented_forward(x_in):
        feat     = model.encoder(x_in)
        heatmaps = {}
        for l in [1, 2, 3, 4]:
            if str(l) in model.proto_layers and l not in model.pruned_levels:
                heatmaps[l] = model._proto_layer(l)(feat[l])

        w = model.level_attention(feat) if model.level_attention is not None else None

        masked = {}
        for l in [1, 2, 3, 4]:
            if str(l) in model.proto_layers and l not in model.pruned_levels:
                A = heatmaps[l]
                if w is not None:
                    A_blended = torch.zeros_like(A[:, :, 0, :, :])
                    w_sum = torch.zeros(A.shape[0], 1, 1, 1, device=A.device)
                    for j, l2 in enumerate(model.proto_levels):
                        if l2 in model.pruned_levels: continue
                        A_l2_up = F.interpolate(
                            heatmaps[l2].max(dim=2).values,
                            size=A.shape[-2:], mode="bilinear", align_corners=False)
                        A_blended += w[:, j:j+1, None, None] * A_l2_up
                        w_sum     += w[:, j:j+1, None, None]
                    agg = (A_blended / (w_sum + 1e-8)).unsqueeze(2)
                else:
                    agg = A

                # Intercept feat[l] as leaf
                f = feat[l].detach().requires_grad_(True)
                feat_tensors[l] = f

                # Intercept aggregated heatmap as leaf
                h = (agg.max(dim=2).values if agg.dim() == 5 else agg)
                h = h.detach().requires_grad_(True)
                heatmap_tensors[l] = h

                # Simplified SoftMask
                masked[l] = f * h.mean(dim=1, keepdim=True)
            elif l not in model.pruned_levels:
                masked[l] = feat[l]
            else:
                masked[l] = torch.zeros_like(feat[l])

        d = model.dec4(masked[4], masked[3])
        d = model.dec3(d, masked[2])
        d = model.dec2(d, masked[1])
        d = F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False)
        d = model.dec1(d)
        return model.final_conv(d), heatmaps

    model.forward = instrumented_forward
    try:
        logits, _ = model(x)
    finally:
        model.forward = original_forward

    logits[0, target_class].sum().backward()

    results = {}
    for l in feat_tensors:
        gf = feat_tensors[l].grad
        gh = heatmap_tensors[l].grad if l in heatmap_tensors else None
        nf = gf.norm().item() if gf is not None else 0.0
        nh = gh.norm().item() if gh is not None else 0.0
        results[l] = {
            "grad_feat":    nf,
            "grad_heatmap": nh,
            "bypass_ratio": nf / (nf + nh + 1e-12),
        }
    return results

## 3. Run Attribution

In [ ]:
def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    m = ProtoSegNet(
        proto_levels        = ckpt.get("proto_levels", None),
        single_scale        = ckpt.get("single_scale", False),
        hard_mask           = ckpt.get("hard_mask", False),
        no_soft_mask        = ckpt.get("no_soft_mask", False),
        use_level_attention = ckpt.get("use_level_attention", False),
    )
    m.load_state_dict(ckpt["model_state_dict"])
    return m.to(DEVICE).eval()


def run_attribution(model_name, ckpt_path):
    # Skip if already computed
    out_path = OUT_DIR / f"gradient_attribution_{model_name}.csv"
    if out_path.exists():
        print(f"  {model_name}: loading cached results")
        return pd.read_csv(out_path)

    print(f"\nRunning: {model_name}")
    model = load_model(ckpt_path)
    print(f"  Active levels: {sorted(model.proto_layers_dict().keys())}")

    rows = []
    for slice_idx, batch in enumerate(loader):
        if slice_idx % 10 == 0: print(f"  Slice {slice_idx}/{len(loader)}...")
        x = batch["image"].float()
        for k in FG_CLASSES:
            try:
                attr = compute_attribution_for_slice(model, x, target_class=k)
            except Exception as e:
                print(f"    Warning: slice {slice_idx} class {k}: {e}")
                continue
            for l, vals in attr.items():
                rows.append({
                    "model": model_name, "slice_idx": slice_idx,
                    "class_idx": k, "class_name": CLASS_NAMES[k], "level": l,
                    **vals,
                })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print(f"  Saved ({len(df)} rows)")
    return df


dfs = {}
for model_name, ckpt_path in MODELS.items():
    dfs[model_name] = run_attribution(model_name, ckpt_path)

print("\nAll models loaded.")

## 4. Core 2×2 Comparison

Controlled comparison: same level configuration, skip vs no-skip.
No-skip models (9a, 9b) have bypass_ratio = 0 by construction — included as reference.

In [ ]:
# Known AP from v10 2x2 ablation
KNOWN_AP = {
    "stage8a":  0.057,
    "9a":       0.312,
    "stage29":  0.051,
    "9b":       0.301,
    "stage34b": 0.221,
}

print(f"{'Model':<20} {'Config':<18} {'bypass_ratio':>14} {'AP':>8}")
print("-" * 64)

# L4-only pair
br8a = dfs["stage8a"]["bypass_ratio"].mean()
print(f"{'Stage 8A (skip)':<20} {'L4 only':<18} {br8a:>14.3f} {KNOWN_AP['stage8a']:>8.3f}")
print(f"{'9a (no-skip)':<20} {'L4 only':<18} {'0.000 (structural)':>14} {KNOWN_AP['9a']:>8.3f}")
print()

# L3+L4 pair
br29 = dfs["stage29"]["bypass_ratio"].mean()
print(f"{'Stage 29 (skip)':<20} {'L3+L4':<18} {br29:>14.3f} {KNOWN_AP['stage29']:>8.3f}")
print(f"{'9b (no-skip)':<20} {'L3+L4':<18} {'0.000 (structural)':>14} {KNOWN_AP['9b']:>8.3f}")
print()

# ALC
br34b = dfs["stage34b"]["bypass_ratio"].mean()
print(f"{'Stage 34b (ALC)':<20} {'L3+L4, ALC':<18} {br34b:>14.3f} {KNOWN_AP['stage34b']:>8.3f}")

print()
print("Per-level breakdown (skip models):")
for name, df in dfs.items():
    per_l = df.groupby("level")["bypass_ratio"].mean()
    levels_str = "  ".join([f"L{l}={v:.3f}" for l, v in per_l.items()])
    print(f"  {name:<12}: {levels_str}")

## 5. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Left: bypass_ratio vs AP (all models) ────────────────────────────────────
ax = axes[0]
plot_data = [
    ("Stage 8A\n(skip, L4)",  br8a,  KNOWN_AP["stage8a"],  "steelblue",  "o"),
    ("9a\n(no-skip, L4)",     0.0,   KNOWN_AP["9a"],       "steelblue",  "s"),
    ("Stage 29\n(skip, L3+4)",br29,  KNOWN_AP["stage29"],  "darkorange", "o"),
    ("9b\n(no-skip, L3+4)",   0.0,   KNOWN_AP["9b"],       "darkorange", "s"),
    ("Stage 34b\n(ALC)",      br34b, KNOWN_AP["stage34b"], "green",      "^"),
]
for label, br, ap, color, marker in plot_data:
    ax.scatter(br, ap, color=color, marker=marker, s=100, zorder=3)
    ax.annotate(label, (br, ap), textcoords="offset points",
                xytext=(5, 5), fontsize=7)
ax.set_xlabel("Bypass Ratio")
ax.set_ylabel("AP")
ax.set_title("bypass_ratio vs AP")
ax.set_xlim(-0.05, 1.0)
ax.set_ylim(0, 0.4)
ax.axvline(0, color="gray", linestyle="--", alpha=0.4, label="no-skip (structural)")
ax.grid(alpha=0.3)

# ── Middle: L4-only pair ─────────────────────────────────────────────────────
ax = axes[1]
models_l4  = ["Stage 8A\n(skip)", "9a\n(no-skip)"]
bypass_l4  = [br8a, 0.0]
ap_l4      = [KNOWN_AP["stage8a"], KNOWN_AP["9a"]]
x_pos = [0, 1]
bars = ax.bar(x_pos, bypass_l4, color=["steelblue", "lightblue"], alpha=0.8, label="bypass_ratio")
ax2 = ax.twinx()
ax2.plot(x_pos, ap_l4, "ro-", linewidth=2, markersize=8, label="AP")
ax.set_xticks(x_pos)
ax.set_xticklabels(models_l4)
ax.set_ylabel("Bypass Ratio", color="steelblue")
ax2.set_ylabel("AP", color="red")
ax.set_ylim(0, 1)
ax2.set_ylim(0, 0.4)
ax.set_title("L4-only: bypass↑ → AP↓")
for bar, val in zip(bars, bypass_l4):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f"{val:.3f}", ha="center", fontsize=9, color="steelblue")
for xi, val in zip(x_pos, ap_l4):
    ax2.text(xi, val + 0.01, f"{val:.3f}", ha="center", fontsize=9, color="red")
ax.grid(axis="y", alpha=0.3)

# ── Right: L3+L4 pair ────────────────────────────────────────────────────────
ax = axes[2]
models_l34 = ["Stage 29\n(skip)", "9b\n(no-skip)"]
bypass_l34 = [br29, 0.0]
ap_l34     = [KNOWN_AP["stage29"], KNOWN_AP["9b"]]
bars = ax.bar(x_pos, bypass_l34, color=["darkorange", "moccasin"], alpha=0.8)
ax2 = ax.twinx()
ax2.plot(x_pos, ap_l34, "ro-", linewidth=2, markersize=8)
ax.set_xticks(x_pos)
ax.set_xticklabels(models_l34)
ax.set_ylabel("Bypass Ratio", color="darkorange")
ax2.set_ylabel("AP", color="red")
ax.set_ylim(0, 1)
ax2.set_ylim(0, 0.4)
ax.set_title("L3+L4: bypass partial (0.47) yet AP still low")
for bar, val in zip(bars, bypass_l34):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f"{val:.3f}", ha="center", fontsize=9, color="darkorange")
for xi, val in zip(x_pos, ap_l34):
    ax2.text(xi, val + 0.01, f"{val:.3f}", ha="center", fontsize=9, color="red")
ax.grid(axis="y", alpha=0.3)

plt.suptitle("Gradient Attribution: Skip Path vs Prototype Path", fontsize=12)
plt.tight_layout()
fig_path = OUT_DIR / "gradient_attribution_fig.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved → {fig_path}")

## 6. Interpretation

### L4-only: clean mechanistic evidence

Stage 8A bypass_ratio = **0.778** — the decoder draws 78% of its gradient from
encoder/skip features, effectively marginalising the prototype heatmap.
Removing skip (9a) forces bypass_ratio = 0 by construction, and AP jumps 5.5×
(0.057 → 0.312). This directly confirms the Bypass Barrier mechanism for L4 models.

### L3+L4: partial bypass, but AP still near-zero

Stage 29 bypass_ratio = **0.465** — nearly balanced. Yet AP = 0.051, almost identical
to Stage 8A (AP = 0.057). This reveals an important distinction:

| Metric | What it measures |
|--------|------------------|
| bypass_ratio | Is the output *causally sensitive* to the heatmap? |
| AP | Does the heatmap *activate at the right spatial location*? |

Stage 29 can be causally sensitive to heatmaps (low bypass) while heatmaps remain
spatially misaligned (low AP). The Bypass Barrier for L3+L4 operates through
**spatial misalignment** rather than full causal disconnection.

### ALC: improves AP without reducing bypass

Stage 34b bypass_ratio = **0.656** — unexpectedly *higher* than Stage 29 (0.465),
yet AP is 4× better (0.221 vs 0.051). ALC improves AP by forcing prototype vectors
to learn purer spatial representations (Purity: 0.527 → 0.593), not by reducing
decoder bypass. **bypass_ratio and AP are orthogonal dimensions.**

### Summary for paper

| Evidence | L4-only | L3+L4 |
|----------|---------|-------|
| Bypass mechanism supported by gradient attribution | ✅ Strong (0.778 vs 0.000) | ⚠️ Partial (0.465 vs 0.000) |
| AP drops with bypass | ✅ Clear | ✅ Clear (but mechanism differs) |
| Use in paper | Primary mechanism evidence | Observational (spatial misalignment) |

## 7. Smoke Test

In [ ]:
print("=" * 55)
print("SMOKE TEST")
print("=" * 55)

all_ok = True
for model_name, df in dfs.items():
    has_nan   = df[["grad_feat", "grad_heatmap", "bypass_ratio"]].isna().any().any()
    valid_range = df["bypass_ratio"].between(0, 1).all()
    status    = "✅" if (not has_nan and valid_range) else "❌"
    if has_nan or not valid_range: all_ok = False
    print(f"  {model_name:<12} {len(df):>5} rows | "
          f"bypass={df['bypass_ratio'].mean():.3f}±{df['bypass_ratio'].std():.3f} | {status}")

print()
print("Files:")
for f in sorted(OUT_DIR.glob("gradient_attribution*")):
    print(f"  {f.name}")

print()
print("KEY RESULTS:")
print(f"  Stage 8A  (skip, L4):    bypass={br8a:.3f}  AP=0.057")
print(f"  9a        (no-skip, L4): bypass=0.000       AP=0.312  (+5.5× AP, -78pp bypass)")
print(f"  Stage 29  (skip, L3+4):  bypass={br29:.3f}  AP=0.051")
print(f"  9b        (no-skip, L3+4): bypass=0.000     AP=0.301  (+5.9× AP, -47pp bypass)")
print(f"  Stage 34b (ALC, L3+4):   bypass={br34b:.3f}  AP=0.221")
print()
print("FINDING: bypass_ratio and AP are orthogonal.")
print("  L4: gradient attribution directly supports bypass mechanism.")
print("  L3+L4: bypass is partial; AP failure is spatial misalignment.")
print("  ALC: improves AP via prototype quality, not bypass reduction.")